# 03 — Dataset Preparation v2

Prepare **all currently available** eye-image datasets for a future EfficientNetV2-B0
training run. This notebook does **not** train a model and does **not** modify:

- original source datasets under `dataset/`
- `model/efficientnetv2_finetuned_best.keras`
- notebooks 04–08, Grad-CAM, frontend, backend, or MySQL

Existing project labels remain **Class 0 … Class 4**. Clinical disease names are
**not documented** in this project (see notebook 07 and `reports/severity_analysis/`).
This notebook therefore maps a source folder to a target class **only** when that
mapping is already defined by the project. Unmappable labels are reported as
`UNMAPPED` and are **not** copied into the combined set.


In [ ]:
# STEP 1 — Environment and paths (Windows-compatible)
import hashlib
import json
import os
import re
import shutil
import sys
import warnings
from collections import defaultdict
from datetime import datetime, timezone
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image, UnidentifiedImageError
from sklearn.model_selection import train_test_split

try:
    from IPython.display import display
except ImportError:
    def display(obj):
        print(obj)

warnings.filterwarnings("ignore", category=UserWarning)
try:
    import matplotlib
    if os.environ.get("MPLBACKEND"):
        matplotlib.use(os.environ["MPLBACKEND"])
except Exception:
    pass
plt.rcParams["figure.figsize"] = (9, 4)
plt.rcParams["axes.grid"] = False

print("Python:", sys.version)
print("Executable:", sys.executable)

HARDCODED_ROOT = Path(r"D:/Practice Projects/Disease Detection")
if HARDCODED_ROOT.exists():
    PROJECT_ROOT = HARDCODED_ROOT
else:
    PROJECT_ROOT = Path.cwd().resolve()
    if PROJECT_ROOT.name == "notebooks":
        PROJECT_ROOT = PROJECT_ROOT.parent

DATASET_ROOT = PROJECT_ROOT / "dataset"
PROCESSED_DIR = PROJECT_ROOT / "processed_dataset"
REPORT_DIR = PROJECT_ROOT / "reports" / "dataset"
FIGURE_DIR = REPORT_DIR / "figures"
SPLIT_DIR = PROJECT_ROOT / "preprocessing" / "splits_v2"

for folder in (REPORT_DIR, FIGURE_DIR, SPLIT_DIR):
    folder.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATASET_ROOT:", DATASET_ROOT)
print("PROCESSED_DIR:", PROCESSED_DIR)
print("REPORT_DIR:", REPORT_DIR)
assert DATASET_ROOT.is_dir(), "dataset directory not found"


In [ ]:
# STEP 2 — Project constants (must match notebooks 02 and 04)
SEED = 42
NUM_CLASSES = 5
IMG_SIZE = (224, 224)
TRAIN_FRACTION = 0.70
VAL_FRACTION = 0.15
TEST_FRACTION = 0.15
MIN_SIDE_PX = 32
SUSPICIOUSLY_FEW = 50
IMBALANCE_RATIO = 1.5
SUPPORTED_FORMATS = {"JPEG", "JPG", "PNG"}
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}
SKIP_DIR_NAMES = {
    "masks", "masks_cropped", "masks_square", "gts", "illustrations",
    "models", "semi-automatic-annotations", "labels", "processed_dataset",
    "__pycache__",
}

CLASS_NAMES = {
    0: "Class 0",
    1: "Class 1",
    2: "Class 2",
    3: "Class 3",
    4: "Class 4",
}

print("Existing project mapping:")
for k, v in CLASS_NAMES.items():
    print(f"  {k} -> {v}")
print("Split ratios (same as notebook 02): 70% train / 15% val / 15% test")
print("EfficientNetV2-B0 input:", IMG_SIZE, "RGB, pixels 0-255 (applied only in processed copies / preview)")
print("Original source images will not be overwritten.")


## Automatic dataset discovery

Each first-level folder under `dataset/` is treated as a source collection.
Class folders, YOLO layouts, and CSV-labelled glaucoma collections are detected
without assuming that identically named folders mean the same disease.


In [ ]:
# STEP 3 — Discover every image dataset under dataset/
def list_images(folder: Path) -> list[Path]:
    if not folder.is_dir():
        return []
    return sorted(
        p for p in folder.iterdir()
        if p.is_file() and p.suffix.lower() in IMAGE_EXTS
    )


def count_images_recursive(folder: Path) -> int:
    n = 0
    for dirpath, dirnames, filenames in os.walk(folder):
        dirnames[:] = [d for d in dirnames if d.lower() not in SKIP_DIR_NAMES]
        n += sum(1 for name in filenames if Path(name).suffix.lower() in IMAGE_EXTS)
    return n


def class_like_subdirs(folder: Path) -> list[str]:
    names = []
    for child in sorted(folder.iterdir()):
        if child.is_dir() and child.name.lower() not in SKIP_DIR_NAMES:
            if list_images(child):
                names.append(child.name)
    return names


inventory_rows = []
print("=" * 88)
for dataset_dir in sorted([p for p in DATASET_ROOT.iterdir() if p.is_dir()]):
    class_folders = class_like_subdirs(dataset_dir)
    yolo_splits = []
    for split in ("train", "val", "test"):
        img_dir = dataset_dir / split / "images"
        if img_dir.is_dir():
            yolo_splits.append(f"{split}:{len(list_images(img_dir))}")
    formats = defaultdict(int)
    for dirpath, dirnames, filenames in os.walk(dataset_dir):
        if Path(dirpath).name.lower() in SKIP_DIR_NAMES:
            dirnames[:] = []
            continue
        dirnames[:] = [d for d in dirnames if d.lower() not in SKIP_DIR_NAMES]
        for name in filenames:
            ext = Path(name).suffix.lower()
            if ext in IMAGE_EXTS:
                formats[ext] += 1
    n_images = sum(formats.values())
    row = {
        "dataset_name": dataset_dir.name,
        "path": str(dataset_dir),
        "n_images": n_images,
        "class_folders": ", ".join(class_folders) if class_folders else "",
        "yolo_splits": ", ".join(yolo_splits) if yolo_splits else "",
        "image_formats": ", ".join(f"{k}:{v}" for k, v in sorted(formats.items())),
    }
    inventory_rows.append(row)
    print(f"Dataset: {row['dataset_name']}")
    print(f"  path: {row['path']}")
    print(f"  images (excluding mask/label dirs): {n_images}")
    print(f"  class folders: {row['class_folders'] or '(none)'}")
    if row["yolo_splits"]:
        print(f"  YOLO image splits: {row['yolo_splits']}")
    print(f"  formats: {row['image_formats']}")
    nested = [p.name for p in dataset_dir.iterdir() if p.is_dir()]
    print(f"  top-level subfolders: {nested}")
    print("-" * 88)

inventory_df = pd.DataFrame(inventory_rows)
inventory_df.to_csv(REPORT_DIR / "dataset_inventory.csv", index=False)
print("\nSaved:", REPORT_DIR / "dataset_inventory.csv")
display(inventory_df)


In [ ]:
# STEP 4 — Class / label distribution for every source (raw labels, not remapped)
dist_rows = []


def add_dist(dataset_name, source_label, n, note=""):
    dist_rows.append({
        "dataset_name": dataset_name,
        "source_label": str(source_label),
        "n_images": int(n),
        "note": note,
    })


# Messidor-style class folders
messidor = DATASET_ROOT / "Messidor-2+EyePac_Balanced"
if messidor.is_dir():
    for folder in sorted(p for p in messidor.iterdir() if p.is_dir()):
        add_dist(messidor.name, folder.name, len(list_images(folder)), "class subfolder")

# Named disease folders
eye_named = DATASET_ROOT / "Eye Disease detection"
if eye_named.is_dir():
    for folder in sorted(p for p in eye_named.iterdir() if p.is_dir()):
        add_dist(eye_named.name, folder.name, len(list_images(folder)), "named disease folder")

# YOLO numeric class ids from label files
yolo_root = DATASET_ROOT / "Dataset"
if (yolo_root / "train" / "labels").is_dir():
    yolo_counts = defaultdict(int)
    yolo_unlabelled = 0
    for split in ("train", "val", "test"):
        images = {p.stem: p for p in list_images(yolo_root / split / "images")}
        for stem, img in images.items():
            lab = yolo_root / split / "labels" / f"{stem}.txt"
            ids = []
            if lab.is_file():
                for line in lab.read_text(encoding="utf-8", errors="replace").splitlines():
                    parts = line.strip().split()
                    if parts:
                        ids.append(parts[0])
            if ids:
                yolo_counts[ids[0]] += 1
            else:
                yolo_unlabelled += 1
    for cid, n in sorted(yolo_counts.items(), key=lambda kv: kv[0]):
        add_dist("Dataset", f"yolo_class_{cid}", n, "YOLO bbox class id; not a project Class folder")
    if yolo_unlabelled:
        add_dist("Dataset", "yolo_unlabelled", yolo_unlabelled, "image has no YOLO class id")

# Glaucoma CSV / JSON labels (primary fundus photos only)
g1020_csv = DATASET_ROOT / "GlaucomaFundusImaging" / "G1020" / "G1020.csv"
if g1020_csv.is_file():
    gdf = pd.read_csv(g1020_csv)
    for label, n in gdf["binaryLabels"].value_counts().sort_index().items():
        add_dist("G1020", f"binaryLabels={label}", n, "G1020.csv binary glaucoma label")

origa_csv = DATASET_ROOT / "GlaucomaFundusImaging" / "ORIGA" / "OrigaList.csv"
if origa_csv.is_file():
    odf = pd.read_csv(origa_csv)
    col = "Glaucoma" if "Glaucoma" in odf.columns else odf.columns[-1]
    for label, n in odf[col].value_counts().sort_index().items():
        add_dist("ORIGA", f"Glaucoma={label}", n, "OrigaList.csv binary glaucoma label")

refuge_root = DATASET_ROOT / "GlaucomaFundusImaging" / "REFUGE"
for split in ("train", "val", "test"):
    idx = refuge_root / split / "index.json"
    if not idx.is_file():
        continue
    payload = json.loads(idx.read_text(encoding="utf-8"))
    counts = defaultdict(int)
    for item in payload.values():
        if "Label" in item:
            counts[item["Label"]] += 1
        else:
            counts["unlabelled"] += 1
    for label, n in counts.items():
        add_dist("REFUGE", f"{split}:Label={label}", n, "REFUGE index.json glaucoma label")

dist_df = pd.DataFrame(dist_rows)
dist_df.to_csv(REPORT_DIR / "source_class_distribution.csv", index=False)
print("Raw source-label distribution (NOT project Class 0-4):")
display(dist_df)


## Existing project class mapping

Notebook 04, notebook 07, and `reports/severity_analysis/class_label_inventory.csv`
define the training task as **five integer folders** `0, 1, 2, 3, 4` displayed as
**Class 0 … Class 4**. Clinical names and severity percentages are **not present**.

Therefore:

| Source | Can map? | Reason |
| --- | --- | --- |
| `Messidor-2+EyePac_Balanced` folders `0`–`4` | yes | this **is** the project mapping |
| `Eye Disease detection` `{cataract, diabetic_retinopathy, glaucoma, normal}` | **no** | different taxonomy; mapping them onto Class 0–4 would invent disease names |
| `Dataset` YOLO ids `0`/`1` | **no** | detection ids, 2 classes, names unknown; must not assume they equal project classes |
| G1020 / ORIGA / REFUGE binary glaucoma labels | **no** | binary glaucoma task, not 5-class project labels |

Unmapped sources are inspected and validated, then **excluded** from the combined
processed dataset.


In [ ]:
# STEP 5 — Explicit mapping configuration (only project-supported mappings)
# Keys are (dataset_key, source_label) -> target class id.
# dataset_key is the top-level collection or named sub-collection used in dist tables.

SUPPORTED_MAPPING = {
    ("Messidor-2+EyePac_Balanced", "0"): 0,
    ("Messidor-2+EyePac_Balanced", "1"): 1,
    ("Messidor-2+EyePac_Balanced", "2"): 2,
    ("Messidor-2+EyePac_Balanced", "3"): 3,
    ("Messidor-2+EyePac_Balanced", "4"): 4,
}

mapping_table = []
for (dataset_name, source_label), n in dist_df.groupby(["dataset_name", "source_label"])["n_images"].sum().items():
    mapped = SUPPORTED_MAPPING.get((dataset_name, str(source_label)))
    mapping_table.append({
        "dataset_name": dataset_name,
        "source_label": str(source_label),
        "n_images": int(n),
        "target_class": mapped if mapped is not None else "UNMAPPED",
        "target_label": CLASS_NAMES[mapped] if mapped is not None else "UNMAPPED",
        "mapping_supported_by_project": mapped is not None,
        "reason": (
            "Existing Messidor-2+EyePac_Balanced integer folder used by notebooks 01-04."
            if mapped is not None
            else "Source label is not defined as Class 0-4 in this project. Not assigned."
        ),
    })

mapping_df = pd.DataFrame(mapping_table).sort_values(["mapping_supported_by_project", "dataset_name", "source_label"], ascending=[False, True, True])
mapping_df.to_csv(REPORT_DIR / "mapping_configuration.csv", index=False)
unmapped_df = mapping_df[mapping_df["target_class"] == "UNMAPPED"].copy()
unmapped_df.to_csv(REPORT_DIR / "unmapped_classes.csv", index=False)

print("Mapping configuration")
display(mapping_df)
print("\nUNMAPPED labels (will not be combined):")
display(unmapped_df if len(unmapped_df) else pd.DataFrame({"note": ["none"]}))

HAS_UNMAPPED = len(unmapped_df) > 0
print("HAS_UNMAPPED:", HAS_UNMAPPED)


In [ ]:
# STEP 6 — Collect every classification-candidate image with its source label
records = []


def append_record(dataset_name, source_label, path: Path, note=""):
    records.append({
        "dataset_name": dataset_name,
        "source_label": str(source_label),
        "original_image_path": str(path),
        "filename": path.name,
        "note": note,
    })


if messidor.is_dir():
    for folder in sorted(p for p in messidor.iterdir() if p.is_dir()):
        for img in list_images(folder):
            append_record(messidor.name, folder.name, img, "class subfolder")

if eye_named.is_dir():
    for folder in sorted(p for p in eye_named.iterdir() if p.is_dir()):
        for img in list_images(folder):
            append_record(eye_named.name, folder.name, img, "named disease folder")

if (yolo_root / "train" / "images").is_dir():
    for split in ("train", "val", "test"):
        images = list_images(yolo_root / split / "images")
        for img in images:
            lab = yolo_root / split / "labels" / f"{img.stem}.txt"
            source_label = "yolo_unlabelled"
            if lab.is_file():
                for line in lab.read_text(encoding="utf-8", errors="replace").splitlines():
                    parts = line.strip().split()
                    if parts:
                        source_label = f"yolo_class_{parts[0]}"
                        break
            append_record("Dataset", source_label, img, f"YOLO {split} image")

if g1020_csv.is_file():
    gdf = pd.read_csv(g1020_csv)
    img_dir = DATASET_ROOT / "GlaucomaFundusImaging" / "G1020" / "Images"
    for _, row in gdf.iterrows():
        img = img_dir / str(row["imageID"])
        if img.is_file():
            append_record("G1020", f"binaryLabels={row['binaryLabels']}", img, "G1020 primary fundus photo")

if origa_csv.is_file():
    odf = pd.read_csv(origa_csv)
    img_dir = DATASET_ROOT / "GlaucomaFundusImaging" / "ORIGA" / "Images"
    col = "Glaucoma" if "Glaucoma" in odf.columns else odf.columns[-1]
    name_col = "Filename" if "Filename" in odf.columns else odf.columns[1]
    for _, row in odf.iterrows():
        img = img_dir / str(row[name_col])
        if img.is_file():
            append_record("ORIGA", f"Glaucoma={row[col]}", img, "ORIGA primary fundus photo")

for split in ("train", "val", "test"):
    idx = refuge_root / split / "index.json"
    img_dir = refuge_root / split / "Images"
    if not idx.is_file() or not img_dir.is_dir():
        continue
    payload = json.loads(idx.read_text(encoding="utf-8"))
    for item in payload.values():
        name = item.get("ImgName")
        if not name:
            continue
        img = img_dir / name
        if not img.is_file():
            continue
        if "Label" in item:
            source_label = f"{split}:Label={item['Label']}"
        else:
            source_label = f"{split}:Label=unlabelled"
        append_record("REFUGE", source_label, img, f"REFUGE {split} fundus photo")

all_df = pd.DataFrame(records)
print("Collected classification-candidate images:", len(all_df))
print(all_df.groupby(["dataset_name", "source_label"]).size())


In [ ]:
# STEP 7 — Quality scan without modifying source filesquality_rows = []
for idx, row in all_df.iterrows():
    path = Path(row["original_image_path"])
    status = "valid"
    reason = ""
    width = height = None
    fmt = None
    mode = None
    sha256 = None
    phash = None
    mapped = SUPPORTED_MAPPING.get((row["dataset_name"], str(row["source_label"])))
    try:
        data = path.read_bytes()
        sha256 = hashlib.sha256(data).hexdigest()
        if not data:
            status, reason = "invalid", "empty file"
        else:
            with Image.open(path) as im:
                im.verify()
            with Image.open(path) as im:
                width, height = im.size
                fmt = im.format
                mode = im.mode
            if width is None or height is None or min(width, height) < MIN_SIDE_PX:
                status, reason = "invalid", f"too small ({width}x{height})"
            elif fmt not in SUPPORTED_FORMATS:
                status, reason = "unsupported_format", f"format {fmt}"
    except (UnidentifiedImageError, OSError, ValueError) as exc:
        status, reason = "corrupted", type(exc).__name__

    quality_rows.append({
        **row,
        "width": width,
        "height": height,
        "image_format": fmt,
        "mode": mode,
        "sha256": sha256,
        "phash": phash,
        "quality_status": status,
        "quality_reason": reason,
        "target_class": mapped if mapped is not None else pd.NA,
        "target_label": CLASS_NAMES[mapped] if mapped is not None else "UNMAPPED",
        "mapped": mapped is not None,
    })
    if (idx + 1) % 2000 == 0:
        print(f"scanned {idx + 1}/{len(all_df)}")

quality_df = pd.DataFrame(quality_rows)
print("scan complete:", len(quality_df))


In [ ]:
# STEP 8 — Duplicate groups: exact SHA-256, plus Windows " - Copy" filename families
# Perceptual hashing is not used: 8x8 hashes collide heavily on fundus photographs
# and would incorrectly glue different images into one split.
COPY_SUFFIX = re.compile(r"(\s*-\s*copy(?:\s*\(\d+\))?)+$", re.I)


def related_stem(filename: str) -> str:
    stem = Path(filename).stem
    stem = COPY_SUFFIX.sub("", stem).strip().lower()
    return stem


quality_df["duplicate_group"] = pd.NA
group_id = 0
sha_groups = quality_df.dropna(subset=["sha256"]).groupby("sha256").indices
for sha, idxs in sha_groups.items():
    if len(idxs) > 1:
        group_id += 1
        quality_df.loc[list(idxs), "duplicate_group"] = f"sha_{group_id:05d}"

quality_df["related_stem"] = quality_df["filename"].map(related_stem)
family = quality_df[quality_df["duplicate_group"].isna()].groupby(
    ["dataset_name", "source_label", "related_stem"]
).indices
for key, idxs in family.items():
    if len(idxs) > 1:
        group_id += 1
        quality_df.loc[list(idxs), "duplicate_group"] = f"copy_{group_id:05d}"

quality_df["is_duplicate"] = quality_df["duplicate_group"].notna()
quality_df["split_group"] = quality_df["duplicate_group"]
unique_mask = quality_df["split_group"].isna()
quality_df.loc[unique_mask, "split_group"] = [
    f"unique_{k}" for k in range(int(unique_mask.sum()))
]
print("duplicate clusters:", quality_df["duplicate_group"].nunique(dropna=True))
print("duplicate image rows:", int(quality_df["is_duplicate"].sum()))


In [ ]:
# STEP 9 — Per-dataset validation report
def report_for(df: pd.DataFrame) -> pd.Series:
    return pd.Series({
        "n_collected": len(df),
        "valid": int(df["quality_status"].eq("valid").sum()),
        "invalid": int(df["quality_status"].eq("invalid").sum()),
        "corrupted": int(df["quality_status"].eq("corrupted").sum()),
        "unsupported_format": int(df["quality_status"].eq("unsupported_format").sum()),
        "duplicate_rows": int(df["is_duplicate"].sum()),
        "mapped": int(df["mapped"].sum()),
        "unmapped": int((~df["mapped"]).sum()),
        "usable": int(((df["quality_status"] == "valid") & df["mapped"]).sum()),
    })


rows = []
for name, group in quality_df.groupby("dataset_name", dropna=False):
    rec = report_for(group)
    rec["dataset_name"] = name
    rows.append(rec)
validation_df = pd.DataFrame(rows)
overall = report_for(quality_df)
overall["dataset_name"] = "ALL"
validation_df = pd.concat([validation_df, overall.to_frame().T], ignore_index=True)
cols = ["dataset_name", "n_collected", "valid", "invalid", "corrupted", "unsupported_format", "duplicate_rows", "mapped", "unmapped", "usable"]
validation_df = validation_df[cols]
validation_df.to_csv(REPORT_DIR / "validation_report.csv", index=False)
quality_df.to_csv(REPORT_DIR / "image_quality_scan.csv", index=False, encoding="utf-8")
print("Validation report")
display(validation_df)


## Combine, split, and write processed copies

Only **mapped + valid** images are eligible. Unmapped sources stay in the reports
and are never copied into `processed_dataset/`. Source trees under `dataset/` are
not modified.


In [ ]:
# STEP 10 — Eligible mapped images; block combining of UNMAPPED rows
eligible_df = quality_df[
    (quality_df["quality_status"] == "valid") & (quality_df["mapped"] == True)
].copy()
eligible_df["target_class"] = eligible_df["target_class"].astype(int)
eligible_df["target_label"] = eligible_df["target_class"].map(CLASS_NAMES)

print("Eligible mapped+valid images:", len(eligible_df))
print(eligible_df["target_label"].value_counts().sort_index())
print("Unmapped rows excluded from combine:", int((~quality_df["mapped"]).sum()))

block_reasons = []
if HAS_UNMAPPED:
    block_reasons.append(
        "One or more source datasets have labels that are not defined as Class 0-4 "
        "in this project. They were not combined. See unmapped_classes.csv."
    )
if eligible_df.empty:
    block_reasons.append("No mapped+valid images remain after quality filtering.")

class_counts = eligible_df["target_class"].value_counts() if len(eligible_df) else pd.Series(dtype=int)
for class_id in range(NUM_CLASSES):
    n = int(class_counts.get(class_id, 0))
    if n == 0:
        block_reasons.append(f"{CLASS_NAMES[class_id]} has zero usable mapped images.")
    elif n < SUSPICIOUSLY_FEW:
        block_reasons.append(
            f"{CLASS_NAMES[class_id]} has only {n} usable images (threshold {SUSPICIOUSLY_FEW})."
        )

print("Preliminary block reasons:", block_reasons or ["none yet"])


In [ ]:
# STEP 11 — Stratified 70/15/15 split with duplicate groups kept together
metadata_df = eligible_df.copy()
if len(metadata_df) == 0:
    raise RuntimeError("No eligible images to split.")

# If a hash group mixes target classes, keep it out of training rather than leaking labels.
mixed = (
    metadata_df.groupby("split_group")["target_class"].nunique()
)
mixed_groups = mixed[mixed > 1].index.tolist()
if mixed_groups:
    block_reasons.append(f"{len(mixed_groups)} duplicate groups mix more than one target class.")
    metadata_df = metadata_df[~metadata_df["split_group"].isin(mixed_groups)].copy()
    print("Removed mixed-label duplicate groups:", len(mixed_groups))

group_frame = (
    metadata_df.groupby("split_group")
    .agg(target_class=("target_class", "first"), n=("original_image_path", "size"))
    .reset_index()
)

train_g, temp_g = train_test_split(
    group_frame,
    test_size=(VAL_FRACTION + TEST_FRACTION),
    stratify=group_frame["target_class"],
    random_state=SEED,
)
val_g, test_g = train_test_split(
    temp_g,
    test_size=TEST_FRACTION / (VAL_FRACTION + TEST_FRACTION),
    stratify=temp_g["target_class"],
    random_state=SEED,
)

split_lookup = {}
for name, frame in (("train", train_g), ("validation", val_g), ("test", test_g)):
    for gid in frame["split_group"]:
        split_lookup[gid] = name

metadata_df["split"] = metadata_df["split_group"].map(split_lookup)
assert metadata_df["split"].notna().all()

# No hash/group overlap across splits
for left, right in (("train", "validation"), ("train", "test"), ("validation", "test")):
    overlap = set(metadata_df.loc[metadata_df["split"] == left, "split_group"]) & set(
        metadata_df.loc[metadata_df["split"] == right, "split_group"]
    )
    if overlap:
        raise RuntimeError(f"Leakage between {left} and {right}: {len(overlap)} groups")

print("Split sizes (images):")
print(metadata_df["split"].value_counts())
print("\nPer-split class counts:")
split_counts = (
    metadata_df.pivot_table(index="target_label", columns="split", values="filename", aggfunc="count")
    .reindex([CLASS_NAMES[i] for i in range(NUM_CLASSES)])
    .fillna(0)
    .astype(int)
)
display(split_counts)
split_counts.to_csv(REPORT_DIR / "split_class_distribution.csv")


In [ ]:
# STEP 12 — Imbalance check; balance TRAIN only if needed
def class_histogram(df, title):
    counts = df["target_class"].value_counts().reindex(range(NUM_CLASSES)).fillna(0).astype(int)
    print(title)
    print(counts.rename(index=CLASS_NAMES))
    return counts


train_df = metadata_df[metadata_df["split"] == "train"].copy()
val_df = metadata_df[metadata_df["split"] == "validation"].copy()
test_df = metadata_df[metadata_df["split"] == "test"].copy()

before_counts = class_histogram(train_df, "TRAIN before balancing")
ratio = float(before_counts.max() / max(int(before_counts.min()), 1))
print(f"Train imbalance ratio (max/min): {ratio:.3f}")

train_balanced = train_df.copy()
balanced_by = "none"
if ratio >= IMBALANCE_RATIO:
    target_n = int(before_counts.max())
    parts = []
    rng = np.random.default_rng(SEED)
    for class_id, n in before_counts.items():
        subset = train_df[train_df["target_class"] == class_id]
        if n == 0:
            parts.append(subset)
            continue
        extra_idx = rng.choice(subset.index, size=target_n - n, replace=True)
        parts.append(pd.concat([subset, subset.loc[extra_idx]], axis=0))
    train_balanced = pd.concat(parts, ignore_index=True)
    balanced_by = "random oversample of minority train classes only"
else:
    print("No train oversampling (ratio below threshold). Validation/test left unchanged.")

after_counts = class_histogram(train_balanced, "TRAIN after balancing")

fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharey=True)
axes[0].bar([CLASS_NAMES[i] for i in range(NUM_CLASSES)], before_counts.values, color="#0f6a5c")
axes[0].set_title("Train class counts before balancing")
axes[0].set_ylabel("images")
axes[1].bar([CLASS_NAMES[i] for i in range(NUM_CLASSES)], after_counts.values, color="#3d8b80")
axes[1].set_title("Train class counts after balancing")
for ax in axes:
    ax.tick_params(axis="x", rotation=20)
fig.tight_layout()
fig.savefig(FIGURE_DIR / "train_balancing_class_distribution.png", dpi=140, bbox_inches="tight")
plt.show()

balance_report = pd.DataFrame({
    "target_class": range(NUM_CLASSES),
    "target_label": [CLASS_NAMES[i] for i in range(NUM_CLASSES)],
    "train_before": before_counts.values,
    "train_after": after_counts.values,
    "validation": val_df["target_class"].value_counts().reindex(range(NUM_CLASSES)).fillna(0).astype(int).values,
    "test": test_df["target_class"].value_counts().reindex(range(NUM_CLASSES)).fillna(0).astype(int).values,
    "method": balanced_by,
})
balance_report.to_csv(REPORT_DIR / "balancing_report.csv", index=False)
display(balance_report)


In [ ]:
# STEP 13 — Write processed copies (mapped valid images only). Do not touch source files.
if PROCESSED_DIR.exists():
    shutil.rmtree(PROCESSED_DIR)
for class_id in range(NUM_CLASSES):
    (PROCESSED_DIR / CLASS_NAMES[class_id]).mkdir(parents=True, exist_ok=True)

processed_paths = []
for row in metadata_df.itertuples(index=False):
    src = Path(row.original_image_path)
    dest_name = f"{row.dataset_name}__{row.split}__{str(row.sha256)[:10]}__{src.stem}{src.suffix.lower()}"
    dest_name = dest_name.replace(" ", "_")
    dest = PROCESSED_DIR / row.target_label / dest_name
    shutil.copy2(src, dest)
    processed_paths.append(str(dest))

metadata_df = metadata_df.copy()
metadata_df["processed_image_path"] = processed_paths
print("Copied mapped images into", PROCESSED_DIR)
for class_id in range(NUM_CLASSES):
    folder = PROCESSED_DIR / CLASS_NAMES[class_id]
    print(f"  {folder.name}: {len(list(folder.glob('*')))} files")


In [ ]:
# STEP 14 — Demonstrate EfficientNetV2-B0 224x224x3 RGB 0-255 WITHOUT overwriting originals
def load_224(path: Path) -> np.ndarray:
    with Image.open(path) as im:
        rgb = im.convert("RGB").resize(IMG_SIZE, Image.Resampling.BILINEAR)
        arr = np.asarray(rgb, dtype=np.float32)
    if arr.min() < 0 or arr.max() > 255:
        raise ValueError("Pixel range must stay 0-255")
    return arr


preview = metadata_df.groupby("target_class", group_keys=False).head(1)
fig, axes = plt.subplots(1, NUM_CLASSES, figsize=(14, 3.2))
for ax, row in zip(axes, preview.itertuples(index=False)):
    arr = load_224(Path(row.original_image_path))
    ax.imshow(arr.astype(np.uint8))
    ax.set_title(f"{row.target_label}\n224x224 preview")
    ax.axis("off")
fig.suptitle("In-memory bilinear resize (originals unchanged)")
fig.tight_layout()
fig.savefig(FIGURE_DIR / "input_size_224_preview.png", dpi=140, bbox_inches="tight")
plt.show()
print("Preview arrays are 224x224x3 float32 in 0-255. Source files were not resized on disk.")


In [ ]:
# STEP 15 — Visual samples from every target class and each source dataset
fig, axes = plt.subplots(NUM_CLASSES, 4, figsize=(12, 12))
for class_id in range(NUM_CLASSES):
    subset = metadata_df[metadata_df["target_class"] == class_id]
    sample = subset.sample(min(4, len(subset)), random_state=SEED)
    for col in range(4):
        ax = axes[class_id, col]
        ax.axis("off")
        if col < len(sample):
            row = sample.iloc[col]
            with Image.open(row["original_image_path"]) as im:
                ax.imshow(im.convert("RGB"))
            ax.set_title(f"{row['target_label']} | {row['dataset_name']}", fontsize=8)
fig.suptitle("Representative original images by target class")
fig.tight_layout()
fig.savefig(FIGURE_DIR / "samples_by_target_class.png", dpi=140, bbox_inches="tight")
plt.show()

sources = sorted(metadata_df["dataset_name"].unique())
cols = min(4, max(len(sources), 1))
fig, axes = plt.subplots(1, cols, figsize=(3.2 * cols, 3.4))
if cols == 1:
    axes = [axes]
for ax, src_name in zip(axes, sources):
    row = metadata_df[metadata_df["dataset_name"] == src_name].iloc[0]
    with Image.open(row["original_image_path"]) as im:
        ax.imshow(im.convert("RGB"))
    ax.set_title(src_name, fontsize=8)
    ax.axis("off")
fig.suptitle("One sample from each mapped source dataset")
fig.tight_layout()
fig.savefig(FIGURE_DIR / "samples_by_source_dataset.png", dpi=140, bbox_inches="tight")
plt.show()


In [ ]:
# STEP 16 — Final metadata CSV, split CSVs, and dataset summary
meta_out = metadata_df[[
    "dataset_name",
    "original_image_path",
    "processed_image_path",
    "target_class",
    "target_label",
    "filename",
    "width",
    "height",
    "image_format",
    "split",
    "sha256",
    "split_group",
    "source_label",
]].copy()
meta_out["image_dimensions"] = meta_out["width"].astype(str) + "x" + meta_out["height"].astype(str)
meta_out.to_csv(REPORT_DIR / "combined_metadata.csv", index=False, encoding="utf-8")

train_balanced.assign(split="train_balanced")[["original_image_path", "target_class", "target_label", "split"]].to_csv(
    SPLIT_DIR / "train_balanced.csv", index=False
)
# Recreate split frames from metadata after processed paths are attached.
train_df = metadata_df[metadata_df["split"] == "train"].copy()
val_df = metadata_df[metadata_df["split"] == "validation"].copy()
test_df = metadata_df[metadata_df["split"] == "test"].copy()
for name, frame in (("train", train_df), ("validation", val_df), ("test", test_df)):
    frame[["original_image_path", "processed_image_path", "target_class", "target_label"]].to_csv(
        SPLIT_DIR / f"{name}.csv", index=False
    )

source_contrib = (
    metadata_df.groupby(["dataset_name", "target_label"]).size().unstack(fill_value=0)
    .reindex(columns=[CLASS_NAMES[i] for i in range(NUM_CLASSES)], fill_value=0)
)
source_contrib.to_csv(REPORT_DIR / "source_dataset_contribution.csv")

summary = {
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "total_mapped_valid_images": int(len(metadata_df)),
    "train_count": int(len(train_df)),
    "validation_count": int(len(val_df)),
    "test_count": int(len(test_df)),
    "train_balanced_count": int(len(train_balanced)),
    "balancing_method": balanced_by,
    "input_size": "224x224x3",
    "source_datasets_combined": ", ".join(sorted(metadata_df["dataset_name"].unique())),
    "unmapped_datasets_excluded": ", ".join(sorted(unmapped_df["dataset_name"].unique())) if HAS_UNMAPPED else "",
    "has_unmapped_labels": bool(HAS_UNMAPPED),
    "block_reasons": " | ".join(block_reasons) if block_reasons else "",
}
for class_id in range(NUM_CLASSES):
    summary[f"class_{class_id}_count"] = int((metadata_df["target_class"] == class_id).sum())

summary_df = pd.DataFrame([summary])
summary_df.to_csv(REPORT_DIR / "dataset_summary.csv", index=False)

print("Saved reports in", REPORT_DIR)
display(summary_df.T.rename(columns={0: "value"}))
print("\nImages per class in processed_dataset:")
display(metadata_df["target_label"].value_counts().reindex([CLASS_NAMES[i] for i in range(NUM_CLASSES)]))
print("\nSource contribution:")
display(source_contrib)
print("\nSplit counts:")
display(split_counts)


In [ ]:
# STEP 17 — Final gate: READY vs BLOCKED
processed_ok = True
for class_id in range(NUM_CLASSES):
    n = len(list((PROCESSED_DIR / CLASS_NAMES[class_id]).glob("*")))
    if n == 0:
        processed_ok = False
        block_reasons.append(f"processed_dataset/{CLASS_NAMES[class_id]} is empty")

if HAS_UNMAPPED:
    # Combining unmapped labels was skipped by design. Status must be BLOCKED
    # until a project-supported mapping exists for those datasets.
    pass

unique_problems = []
for item in block_reasons:
    if item not in unique_problems:
        unique_problems.append(item)

print("=" * 88)
if unique_problems or not processed_ok:
    print("DATASET PREPARATION BLOCKED — REVIEW REQUIRED")
    print()
    for i, problem in enumerate(unique_problems, start=1):
        print(f"{i}. {problem}")
    print()
    print("Mapped Messidor-2+EyePac_Balanced images were still validated, split, and")
    print("copied into processed_dataset/ for inspection. Unmapped datasets were NOT combined.")
    print("Do not start EfficientNetV2-B0 training on the excluded sources.")
else:
    print("DATASET PREPARATION READY FOR TRAINING")
    print("All five target classes are mapped, valid, split, and present in processed_dataset/.")
print("=" * 88)
print("This notebook did not train EfficientNetV2-B0 and did not modify the existing model.")
